In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime

CATALOG = "olist_dev"
S, G = f"{CATALOG}.silver", f"{CATALOG}.gold"
DIM = f"{G}.dim_customer_scd2"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM} (
  customer_sk BIGINT,
  customer_unique_id STRING,
  zip_prefix STRING,
  city STRING,
  state STRING,
  row_hash STRING,
  effective_from TIMESTAMP,
  effective_to TIMESTAMP,
  is_current BOOLEAN
)
""")

DataFrame[]

In [0]:
def build_source():
    c = spark.table(f"{S}.customers")
    o = spark.table(f"{S}.orders").select("customer_id", "order_purchase_timestamp")
    w = Window.partitionBy("customer_unique_id").orderBy(F.col("order_purchase_timestamp").desc_nulls_last())
    src = (c.join(o, "customer_id", "left")
        .withColumn("rn", F.row_number().over(w))
        .filter("rn = 1")
        .select("customer_unique_id",
                F.col("customer_zip_code_prefix").alias("zip_prefix"),
                F.col("customer_city").alias("city"),
                F.col("customer_state").alias("state"))
        .withColumn("row_hash", F.sha2(F.concat_ws("||",
            F.coalesce("zip_prefix", F.lit("")),
            F.coalesce("city", F.lit("")),
            F.coalesce("state", F.lit(""))), 256)))
    return src

In [0]:
def scd2_merge(src, load_ts):
    ts = load_ts.strftime("%Y-%m-%d %H:%M:%S")
    current = spark.table(DIM).filter("is_current = true")

    changed = (src.alias("s")
        .join(current.alias("t"), F.col("s.customer_unique_id") == F.col("t.customer_unique_id"))
        .filter(F.col("s.row_hash") != F.col("t.row_hash"))
        .select("s.*"))

    staged = (changed.withColumn("merge_key", F.lit(None).cast("string"))
        .unionByName(src.withColumn("merge_key", F.col("customer_unique_id")))
        .withColumn("customer_sk", F.xxhash64(F.col("customer_unique_id"), F.lit(ts), F.col("row_hash"))))

    (DeltaTable.forName(spark, DIM).alias("t")
        .merge(staged.alias("s"),
               "t.customer_unique_id = s.merge_key AND t.is_current = true")
        .whenMatchedUpdate(
            condition="t.row_hash <> s.row_hash",
            set={"is_current": "false", "effective_to": f"timestamp'{ts}'"})
        .whenNotMatchedInsert(values={
            "customer_sk": "s.customer_sk",
            "customer_unique_id": "s.customer_unique_id",
            "zip_prefix": "s.zip_prefix",
            "city": "s.city",
            "state": "s.state",
            "row_hash": "s.row_hash",
            "effective_from": f"timestamp'{ts}'",
            "effective_to": "timestamp'9999-12-31 00:00:00'",
            "is_current": "true"})
        .execute())
    print("Dim total rows:", spark.table(DIM).count(),
          "| current:", spark.table(DIM).filter("is_current").count())